# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
# load customers.csv into a DataFrame
# take a first look (head, info)
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt

df = pd.read_csv("./customers.csv")
df.head(20)

,customer_id,age,income,spending,region,subscribed
0,1,25,61.3,39.1,West,No
1,2,25,54.9,42.1,South,Yes
2,3,51,105.1,59.9,South,No
3,4,39,NaN,49.9,South,No
4,5,43,72.4,48.3,East,No
5,6,44,84.7,47.1,East,No
6,7,48,92.9,60.0,East,Yes
7,8,21,53.3,36.0,South,No
8,9,39,71.9,48.9,East,Yes
9,10,25,70.4,42.1,east,Yes


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# HINT 2: one category label is written in the wrong case -> standardize it
# HINT 3: one value is impossible for its column -> treat it as missing, then fill
# tip: check .isna().sum() and the unique values of the category column

#2
fix_region = {
  "east": "East", "west": "West", "north": "North", "south": "South"
}

df["region"] = df["region"].replace(fix_region)
print(df["region"].value_counts())

#3
df["age"] = np.where(df["age"] <= 0, np.nan, df["age"])

df["income"] = np.where(df["income"] <= 0, np.nan, df["income"])

df["spending"] = np.where(df["spending"] <= 0, np.nan, df["spending"])

#1
df["income"] = df["income"].fillna(df["income"].mean())

print(df["income"].isna().sum())
print(df.describe())

region
East     21
West     10
South     9
Name: count, dtype: int64
0
       customer_id        age      income   spending
count    40.000000  40.000000   40.000000  40.000000
mean     20.500000  40.925000   86.602703  54.520000
std      11.690452  11.474358   16.072153   9.377912
min       1.000000  21.000000   53.300000  36.000000
25%      10.750000  32.500000   75.075000  48.750000
50%      20.500000  41.000000   86.251351  53.650000
75%      30.250000  51.000000   97.100000  62.650000
max      40.000000  59.000000  114.500000  74.700000


**Explain each issue:** what you found and how you handled it.

*Your answer: For number 1, I chose the income value since it had a few missing numbers "NA" and I decided to do fillna so I could fill those values.

For number 2, I chose to fix each region by just changing the uncaptilized letters to capitlized letters.

For number 3, I chose to fix every numbered column because why not, and I did this by just making sure there are no negative values.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [ ]:
# data types: income = continuous, spending = continuous
# tool: linear regression
# hint: two continuous variables moving together

stat_x = sm.add_constant(df['income'])
stat_y = df['spending']

model = sm.OLS(stat_y, stat_x).fit()

predictions = model.predict(stat_x)
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.517
Model:                            OLS   Adj. R-squared:                  0.504
Method:                 Least Squares   F-statistic:                     40.70
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           1.72e-07
Time:                        15:54:21   Log-Likelihood:                -131.22
No. Observations:                  40   AIC:                             266.4
Df Residuals:                      38   BIC:                             269.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         18.1803      5.791      3.140      0.0

*Your answer: Somewhat yes, it seems to be about 50%.*


### Q2. Is a customer's region related to whether they subscribed?

In [4]:
# data types: region = categorical, subscribed = categorical
# tool: chi-square test
# hint: two categorical variables -> build a table with pd.crosstab, then test it

crosstab = pd.crosstab(df['region'], df['subscribed'])
print(crosstab)


subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1


*Your answer: I'd say no, because theres not enough evidence. Even though there were more people who said Yes in east, there are way more people in East than compared to South and West.*

### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [8]:
# data types: predicting a continuous outcome from TWO predictors
# tool: linear regression
# hint: remember to add a constant; look at R-squared and each predictor's p-value

stat_x = sm.add_constant(df[['income', 'age']])
stat_y = df['spending']

model = sm.OLS(stat_y, stat_x).fit()

predictions = model.predict(stat_x)
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.522
Model:                            OLS   Adj. R-squared:                  0.496
Method:                 Least Squares   F-statistic:                     20.17
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           1.19e-06
Time:                        16:08:18   Log-Likelihood:                -131.04
No. Observations:                  40   AIC:                             268.1
Df Residuals:                      37   BIC:                             273.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.0369      6.023      3.161      0.0

*Your answer: I'd say so, since it seems that the more older you are with the more income you have, you will be spending more.*


### Q4. Which region has the most customers, and what is the average income overall?

In [10]:
# hint: count a categorical column
# take the mean of a numeric column

print(df.groupby('region')['income'].mean())
average_all_income = df['income'].mean()
print(f"Average income across all regions: {average_all_income}")

region
East     92.766795
South    83.011411
West     76.890270
Name: income, dtype: float64
Average income across all regions: 86.6027027027027


*Your answer: East has the most customers, the average income seems to be 86.60 overall.


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer: I used them based on what the question asked.*


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer: I think I could, since of course I could just show them the data and they could see for themselves*


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [ ]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?

I'll probably do this later, but I need to work on other stuff first.

*Your answer:*


### Push your code to GitHub